# Distributed Checkpointing Tutorial

## Overview

Distributed checkpointing saves model state across multiple files for efficient parallel I/O.

### Learning Objectives
- Implement sharded checkpoint save/load
- Handle checkpoint consistency
- Optimize I/O performance

## 1. Checkpoint Strategies

| Strategy | Pros | Cons |
|----------|------|------|
| Single file | Simple | Memory limited |
| Sharded | Parallel I/O | Coordination needed |
| Async | Non-blocking | Complex state |

In [ ]:
import torch
import torch.nn as nn
import torch.distributed as dist
from pathlib import Path

class DistributedCheckpointer:
    """Simple distributed checkpoint manager."""
    
    def __init__(self, save_dir: str):
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.rank = dist.get_rank() if dist.is_initialized() else 0
    
    def save(self, model: nn.Module, optimizer, step: int):
        """Save sharded checkpoint."""
        ckpt_dir = self.save_dir / f"step_{step}"
        ckpt_dir.mkdir(exist_ok=True)
        
        # Each rank saves its shard
        torch.save({
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'step': step
        }, ckpt_dir / f"rank_{self.rank}.pt")
        
        if dist.is_initialized():
            dist.barrier()
    
    def load(self, model: nn.Module, optimizer, step: int):
        """Load sharded checkpoint."""
        ckpt_path = self.save_dir / f"step_{step}" / f"rank_{self.rank}.pt"
        ckpt = torch.load(ckpt_path, map_location='cpu')
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        return ckpt['step']

## 2. Summary

### Best Practices

1. **Sharded saves**: Each rank saves its portion
2. **Barrier sync**: Ensure all ranks complete before proceeding
3. **Atomic writes**: Use temp files + rename
4. **Cleanup**: Remove old checkpoints to save storage